# MediBot — Milestone 2, Part 2: Embeddings + FAISS index

This notebook takes the cleaned data from Part 1 and turns it into a **searchable vector index** that the Diagnosis Agent will use in Milestone 3.

## The concepts we're about to make concrete

1. **Embedding** — a function that maps text → a fixed-length vector of floats. Similar texts land in nearby points. An embedding is the bridge between how humans ask ("my skin itches and I have a rash") and how computers compare (vector dot products).
2. **Cosine similarity / inner product** — the standard way to measure "how close" two embeddings are. On unit-length vectors, inner product IS cosine similarity.
3. **Why we normalize** — if we L2-normalize every embedding (all vectors on the unit sphere), we can use the fastest FAISS operation (inner product) and get cosine similarity for free.
4. **FAISS** — a library from Meta for *very* fast vector search. The core operation: "given this query vector, find the top-k most similar vectors in the index."
5. **Index type** — `IndexFlatIP` is an exact brute-force index. For 41 docs it's ideal; it gives ground-truth rankings and avoids the accuracy/speed tradeoff that approximate indexes (HNSW, IVF-PQ) impose at million-doc scale.
6. **RAG (Retrieval-Augmented Generation)** — use retrieval to pull in grounded context, then let an LLM generate with that context. The retrieval step is what you're building right now.


In [1]:
import sys, pathlib
# Make the src/ package importable from a notebook in notebooks/
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
from medibot.data import load_medical_data, disease_to_document, normalize_text
from medibot.rag.embeddings import LocalEmbedding
from medibot.rag.vector_store import FaissStore

data = load_medical_data("../data/raw")
print(f"Loaded {len(data.diseases)} diseases, {len(data.symptom_severity)} symptoms with severity weights.")


Loaded 41 diseases, 132 symptoms with severity weights.


## 1. Seeing an embedding

Let's actually look at what an embedding is. We use `BAAI/bge-small-en-v1.5` — a small (~30 MB) but strong-in-2026 embedding model.

- Input: a string.
- Output: a 384-dimensional vector of floats, unit-length.

Don't try to read the numbers — they're learned features, not human-interpretable. The meaning is in the **geometry**, which we demonstrate next.


In [2]:
embedder = LocalEmbedding("BAAI/bge-small-en-v1.5")
print("Embedding dim:", embedder.dim)
print("Provider name:", embedder.name)

sample = "itchy skin and red rash"
vec = embedder.embed([sample])
print(f"\n'{sample}' -> vector of shape {vec.shape}")
print(f"First 8 dims: {vec[0, :8].round(3)}")
print(f"L2 norm: {np.linalg.norm(vec[0]):.4f}  (should be ~1.0 because we asked for normalized embeddings)")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/Users/lkshay/CapstoneProject/src/medibot/rag/embeddings.py:45: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = int(self._model.get_sentence_embedding_dimension())


Embedding dim: 384
Provider name: local:BAAI/bge-small-en-v1.5



'itchy skin and red rash' -> vector of shape (1, 384)
First 8 dims: [-0.023  0.019  0.056  0.032  0.038 -0.015  0.079  0.046]
L2 norm: 1.0000  (should be ~1.0 because we asked for normalized embeddings)


## 2. Geometry check — similar things cluster

This is the sanity check every embedding pipeline should have. If the model is any good, clinically related phrases should score **higher similarity** than unrelated ones.

For unit vectors, cosine similarity = inner product = `a @ b.T`. So a simple matrix multiply gives us every pairwise score.


In [3]:
examples = [
    "my skin itches and has a red rash",        # 0: skin symptoms
    "itching and skin eruptions",                # 1: also skin
    "high fever with chills and body aches",     # 2: systemic febrile
    "persistent fever and muscle pain",          # 3: also systemic
    "sharp chest pain when I breathe",           # 4: cardiac/resp
    "the stock market crashed today",            # 5: unrelated control
]
V = embedder.embed(examples)
sims = V @ V.T

import pandas as pd
pd.DataFrame(sims.round(2), index=[f"{i}. {e[:25]}" for i, e in enumerate(examples)], columns=range(len(examples)))


,0,1,2,3,4,5
0. my skin itches and has a,1.00,0.79,0.67,0.60,0.61,0.50
1. itching and skin eruption,0.79,1.00,0.66,0.63,0.60,0.43
2. high fever with chills an,0.67,0.66,1.00,0.82,0.71,0.52
3. persistent fever and musc,0.60,0.63,0.82,1.00,0.67,0.49
4. sharp chest pain when I b,0.61,0.60,0.71,0.67,1.00,0.53
5. the stock market crashed,0.50,0.43,0.52,0.49,0.53,1.00


**Read the matrix:** the (0,1) cell should be high (both skin). (2,3) should be high (both febrile). (0,5) and (2,5) should be low (control text is unrelated). That's the model "getting" the medical-vs-finance distinction and the skin-vs-fever distinction. If it didn't, we'd pick a different model.


## 3. Build the document corpus

**Choice:** one document per disease, with the rich format from `disease_to_document()`:
- disease name
- full symptom list (readable + token form)
- full description

This puts lexical signal (exact symptom tokens) and semantic signal (prose description) into a single blob — the embedder picks up both.


In [4]:
documents = [disease_to_document(d, data) for d in data.diseases]
metadata = [{"disease": d} for d in data.diseases]

print(f"Built {len(documents)} documents.\n")
print("--- Example document ---")
print(documents[0])


Built 41 documents.

--- Example document ---
Disease: (vertigo) paroymsal positional vertigo.
Common symptoms: headache, loss of balance, nausea, spinning movements, unsteadiness, vomiting.
Description: Benign paroxysmal positional vertigo (BPPV) is one of the most common causes of vertigo — the sudden sensation that you're spinning or that the inside of your head is spinning. Benign paroxysmal positional vertigo causes brief episodes of mild to intense dizziness.
Symptom tokens: headache loss_of_balance nausea spinning_movements unsteadiness vomiting


## 4. Build the FAISS index

`FaissStore.build()` embeds all documents in one batch, creates an `IndexFlatIP` of the right dimension, and keeps parallel arrays of the source documents + metadata.

**Why `IndexFlatIP` here:** 41 docs × 384 floats × 4 bytes ≈ 63 KB. An exact scan at query time is O(41 * 384) = ~16k multiplies. That's sub-millisecond. Approximate indexes solve a problem we don't have.


In [5]:
store = FaissStore(embedder)
store.build(documents, metadata)
print(f"Index built: {store.index.ntotal} vectors, dim {store.embedder.dim}.")


Index built: 41 vectors, dim 384.


## 5. Retrieval sanity check

Plug in realistic user queries and see if the top hit is the disease we'd expect. Two things to notice:

- The **ranking** (did the right disease land in the top-k?)
- The **score gap** between #1 and #2 (a large gap = confident; a small gap = need more info / ask follow-up)


In [6]:
queries = [
    "I have itching and a red rash with patches on my skin",
    "fever with headache and muscle pain",
    "constipation, pain during bowel movements, and bloody stool",
    "excessive thirst, frequent urination, and unexplained weight loss",
    "yellow skin and eyes, with abdominal pain",
]

for q in queries:
    hits = store.search(q, k=3)
    print(f"Q: {q}")
    for h in hits:
        print(f"   {h.score:.3f}  {h.disease}")
    print()


Q: I have itching and a red rash with patches on my skin
   0.735  fungal infection
   0.706  psoriasis
   0.696  impetigo

Q: fever with headache and muscle pain
   0.762  malaria
   0.758  dengue
   0.725  aids

Q: constipation, pain during bowel movements, and bloody stool
   0.780  dimorphic hemmorhoids(piles)
   0.707  typhoid
   0.695  gastroenteritis

Q: excessive thirst, frequent urination, and unexplained weight loss
   0.753  diabetes
   0.736  hypoglycemia
   0.715  hyperthyroidism

Q: yellow skin and eyes, with abdominal pain
   0.801  chronic cholestasis
   0.758  jaundice
   0.754  hepatitis c



## 6. Quantitative eval — recall@k on known mappings

We already know the ground truth (dataset.csv maps diseases → their typical symptoms). So we can build a synthetic eval set: for each disease, generate a query from its symptom list and check whether the index retrieves the right disease in its top-k.

This is the simplest possible **retrieval evaluation**. Later, RAGAS and DeepEval will give us richer metrics (faithfulness, answer relevance), but recall@k is the foundation.


In [7]:
def make_query(symptoms, take=4):
    tokens = [s.replace("_", " ") for s in sorted(symptoms)][:take]
    return "I have " + ", ".join(tokens)

K = 3
hits_at_k = 0
per_disease = []
for d in data.diseases:
    q = make_query(data.disease_symptoms[d])
    results = store.search(q, k=K)
    ranks = [r.disease for r in results]
    hit_rank = ranks.index(d) + 1 if d in ranks else None
    per_disease.append((d, hit_rank, ranks))
    if hit_rank:
        hits_at_k += 1

print(f"Recall@{K} = {hits_at_k}/{len(data.diseases)} = {hits_at_k/len(data.diseases):.2%}")
print("\nDiseases where the correct hit was NOT in top-3:")
for d, rank, ranks in per_disease:
    if rank is None:
        print(f"  {d}  ->  top3: {ranks}")


Recall@3 = 41/41 = 100.00%

Diseases where the correct hit was NOT in top-3:


## 7. Persist the index

Every downstream step (Diagnosis Agent, LangGraph orchestrator, deployed app) should load this index from disk, not rebuild it. Rebuild is slow and non-deterministic across embedder versions.


In [8]:
store.save("../data/embeddings/faiss_bge_small")
print("Saved. Files written:")
for p in sorted(pathlib.Path("../data/embeddings/faiss_bge_small").iterdir()):
    print("  ", p.name, "-", p.stat().st_size, "bytes")


Saved. Files written:
   index.faiss - 63021 bytes
   meta.json - 24361 bytes


## What we have now

- A clean, normalized data bundle (`MedicalData`) with alias fixups
- A provider-abstracted embedder (swap local ↔ Gemini without touching downstream code)
- A persisted FAISS index with per-document metadata and provider fingerprint
- A retrieval recall@3 baseline to protect against future regressions

**Next (Milestone 3):** build the four specialized agents. The Diagnosis Agent will wrap this FAISS store as its retrieval tool; the other three will be dict-lookup tools over `MedicalData`.
